# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarveyWebbs/ML-Basics/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The Content Action Playbook: Ranked Queue
This queue provides decision-support for the editorial team's weekly sprint planning. We use the ML model's outputs to rank content that is most likely to benefit from a refresh.
Archetype -> Action Mapping:
Archetype: High-traffic potential, but severely outdated.
Action: COMPREHENSIVE_REFRESH (Update stats, add new sections, change publish date).
Reason Code: HIGH_VALUE_STALE (Content is >2 years old and hasn't been touched in 12+ months).
Archetype: Recent content that never gained traction.
Action: INVESTIGATE_INTENT (Check Search Console for keyword mismatch).
Reason Code: EARLY_DECAY (Content is mature enough to rank, but visibility is remarkably low).*

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

playbook_query = f"""
    SELECT
        c.content_hash_id,
        c.content_type,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') AS age_days,
        DATE_DIFF('day', COALESCE(c.content_updated_date, c.content_created_date), DATE '2026-03-01') AS days_since_updated,
        SUM(p.gsc_clicks) as current_clicks,

        -- Business Logic: Assigning actions based on observed directional signals
        CASE
            WHEN DATE_DIFF('day', COALESCE(c.content_updated_date, c.content_created_date), DATE '2026-03-01') > 365
                 AND SUM(p.gsc_clicks) > 100 THEN 'COMPREHENSIVE_REFRESH'
            WHEN DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') BETWEEN 90 AND 180
                 AND SUM(p.gsc_clicks) < 10 THEN 'INVESTIGATE_INTENT'
            ELSE 'MONITOR'
        END AS recommended_action,

        CASE
            WHEN DATE_DIFF('day', COALESCE(c.content_updated_date, c.content_created_date), DATE '2026-03-01') > 365
                 AND SUM(p.gsc_clicks) > 100 THEN 'HIGH_VALUE_STALE'
            WHEN DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') BETWEEN 90 AND 180
                 AND SUM(p.gsc_clicks) < 10 THEN 'EARLY_DECAY'
            ELSE 'STABLE'
        END AS reason_code

    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet') p
      ON c.content_hash_id = p.content_hash_id
    WHERE DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') >= 0
    GROUP BY 1, 2, 3, 4
"""
playbook_df = con.sql(playbook_query).df()

action_queue = playbook_df[playbook_df['recommended_action'] != 'MONITOR'].sort_values(by='current_clicks', ascending=False)

print(f"Playbook generated: {len(action_queue)} items require human review.")
display(action_queue[['recommended_action', 'reason_code', 'age_days', 'days_since_updated']].head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Playbook generated: 53152 items require human review.


,recommended_action,reason_code,age_days,days_since_updated
236896,INVESTIGATE_INTENT,EARLY_DECAY,136,-109
249901,INVESTIGATE_INTENT,EARLY_DECAY,136,4
236882,INVESTIGATE_INTENT,EARLY_DECAY,136,-120
236874,INVESTIGATE_INTENT,EARLY_DECAY,136,4
175287,INVESTIGATE_INTENT,EARLY_DECAY,107,-124


## 2. Intended use and limits

*Intended Use:
This playbook is designed for Content Managers and SEO Strategists. It highlights where editorial resources (writer hours) should be spent to maximize ROI. It transforms measured historical data into a prioritized task list.
Limits & Cost/Value Thinking:
Limits: The recommendations are directional, not absolute. The model does not know if a page is tied to a discontinued product.
Cost/Value: Updating a page costs writer time (e.g., $150/hr). Therefore, we only recommend a COMPREHENSIVE_REFRESH on pages that already exhibit a baseline level of historical traction, ensuring the potential upside justifies the editorial cost.*

In [2]:
cost_value_summary = action_queue.groupby('recommended_action').agg(
    pages_to_review=('content_hash_id', 'count'),
    avg_age_days=('age_days', 'mean'),
    total_clicks_at_risk=('current_clicks', 'sum')
).reset_index()

print("--- QUEUE SUMMARY BY EFFORT ---")
display(cost_value_summary.round(1))


--- QUEUE SUMMARY BY EFFORT ---


,recommended_action,pages_to_review,avg_age_days,total_clicks_at_risk
0,INVESTIGATE_INTENT,53152,150.9,24657.0


## 3. Human review + the no-go list

*Human Review Rules:
A human editor must check the URL's actual intent before executing the recommended action. An algorithm cannot read the room; a human must ensure the update aligns with current brand messaging.
The No-Go List (What should NEVER be automated):
Direct Deletion: The model should never automatically 404/delete a page. Low traffic might just mean it serves a niche B2B compliance purpose.
Historical Archives: "2023 Year in Review" reports must not be refreshed into "2024 Year in Review" to trick the model.
Legal / Compliance / PR: Press releases and legal terms must be excluded from automated refresh queues entirely.*

In [3]:
risky_types = ['Press Release', 'News', 'Legal']
nogo_count = action_queue[action_queue['content_type'].isin(risky_types)].shape[0]

print("--- AUTOMATED NO-GO ENFORCEMENT ---")
if nogo_count > 0:
    print(f"Warning: {nogo_count} items in the queue belong to restricted content types.")
    action_queue = action_queue[~action_queue['content_type'].isin(risky_types)]
    print("-> Restricted items have been safely purged from the final export queue.")
else:
    print("Safe: No restricted content types found in the active action queue.")


--- AUTOMATED NO-GO ENFORCEMENT ---
Safe: No restricted content types found in the active action queue.


## 4. Monitoring / retrain triggers

*Monitoring & Retrain Triggers:
To ensure the observed patterns remain accurate, this playbook requires a light monitoring framework. The queue is considered "stale" and the model must be retrained if:
Google Core Update: A confirmed, major algorithmic shift occurs.
Data Drift: The median age of our content portfolio shifts by > 20% (e.g., after a massive site migration or content pruning sprint).
Time Trigger: 6 months have passed. SEO velocity changes quickly; a Q1 model should not govern Q4 content decisions*

In [4]:
baseline_metrics = {
    "model_version": "v1.0-March2026",
    "median_age_days_baseline": float(playbook_df['age_days'].median()),
    "median_staleness_days_baseline": float(playbook_df['days_since_updated'].median()),
    "retrain_recommended_after": "2026-09-01"
}

print("--- MONITORING BASELINES RECORDED ---")
print(json.dumps(baseline_metrics, indent=4))


--- MONITORING BASELINES RECORDED ---
{
    "model_version": "v1.0-March2026",
    "median_age_days_baseline": 201.0,
    "median_staleness_days_baseline": -80.0,
    "retrain_recommended_after": "2026-09-01"
}


## 5. Exports for the paper

*Exports to the Warehouse Directory:
I am exporting the final, human-safe action queue to the outputs directory. I am also exporting the monitoring JSON and a visual figure showing the distribution of staleness across the queue. These artifacts will serve as the foundation for the final capstone paper next week.*

In [ ]:
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

csv_path = 'work/outputs/w07_action_playbook_queue.csv'
action_queue.to_csv(csv_path, index=False)
print(f"Exported Action Queue -> {csv_path}")

json_path = 'work/outputs/w07_monitoring_baselines.json'
with open(json_path, 'w') as f:
    json.dump(baseline_metrics, f, indent=4)
print(f"Exported Metrics Receipt -> {json_path}")

plt.figure(figsize=(8, 5))
plt.hist(action_queue['days_since_updated'], bins=30, color='teal', edgecolor='black')
plt.title('Distribution of Staleness in the Refresh Queue')
plt.xlabel('Days Since Last Update')
plt.ylabel('Number of Pages')
plt.axvline(365, color='red', linestyle='dashed', linewidth=2, label='1-Year Stale Mark')
plt.legend()

fig_path = 'work/figures/staleness_distribution.png'
plt.savefig(fig_path, bbox_inches='tight')
print(f"Exported Figure -> {fig_path}")
plt.show()


Exported Action Queue -> work/outputs/w07_action_playbook_queue.csv
Exported Metrics Receipt -> work/outputs/w07_monitoring_baselines.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.